# 03 - Model Experiments

Trains and compares the two candidate algorithms supported by
`src/ml_pipeline/train.py` (LightGBM and XGBoost) on the synthetic
feature table, evaluates them with `src/ml_pipeline/evaluate.py`,
and previews the responsible AI checks (SHAP explainability and
fairness metrics) that run in the AML pipeline
(`src/ml_pipeline/pipeline_definition.py`).

This notebook mirrors `configs/train_config.yaml` but uses a smaller
dataset and fewer estimators so it runs quickly for experimentation.

In [ ]:
import sys
from datetime import date
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

import pandas as pd

from src.common.config import load_config
from src.common.schemas import CATEGORICAL_COLUMNS, LABEL_COLUMN, MODEL_FEATURE_COLUMNS
from src.data_pipeline.ingest import generate_synthetic_encounters
from src.data_pipeline.transform import transform
from src.ml_pipeline.evaluate import evaluate
from src.ml_pipeline.train import prepare_xy, temporal_split, train_model

pd.set_option("display.max_columns", None)

config = load_config("../../configs/train_config.yaml")

In [ ]:
raw = generate_synthetic_encounters(n_rows=8000, start_date=date(2024, 1, 1), seed=42)
features = transform(raw)

train_df, val_df, test_df = temporal_split(
    features,
    train_fraction=config["data"].get("train_fraction", 0.70),
    validation_fraction=config["data"].get("validation_fraction", 0.15),
)
len(train_df), len(val_df), len(test_df)

## Experiment hyperparameters

Smaller `n_estimators` than `configs/train_config.yaml` for fast
iteration. Swap in the full config values for a final run.

In [ ]:
EXPERIMENT_HYPERPARAMETERS = {
    "lightgbm": {
        "num_leaves": 31,
        "learning_rate": 0.05,
        "n_estimators": 100,
        "min_child_samples": 20,
        "feature_fraction": 0.9,
    },
    "xgboost": {
        "max_depth": 6,
        "learning_rate": 0.05,
        "n_estimators": 100,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
    },
}

results = {}
bundles = {}

for algorithm, hyperparameters in EXPERIMENT_HYPERPARAMETERS.items():
    X_train, y_train = prepare_xy(train_df, algorithm=algorithm)
    X_val, y_val = prepare_xy(val_df, algorithm=algorithm)

    model = train_model(
        X_train, y_train, X_val, y_val,
        algorithm=algorithm,
        hyperparameters=hyperparameters,
        random_seed=config["model"]["random_seed"],
    )

    model_bundle = {
        "model": model,
        "algorithm": algorithm,
        "feature_columns": MODEL_FEATURE_COLUMNS,
        "categorical_columns": CATEGORICAL_COLUMNS,
        "categorical_categories": {c: sorted(train_df[c].unique()) for c in CATEGORICAL_COLUMNS},
        "train_columns": list(X_train.columns),
    }

    bundles[algorithm] = model_bundle
    results[algorithm] = evaluate(model_bundle, test_df, config)

    print(f"{algorithm}: roc_auc={results[algorithm]['roc_auc']:.4f} "
          f"pr_auc={results[algorithm]['pr_auc']:.4f} "
          f"brier_score={results[algorithm]['brier_score']:.4f}")

## Compare algorithms against the promotion gate

Mirrors the checks in `src/ml_pipeline/register_model.py`
(`evaluation.promotion_gate` in `configs/train_config.yaml`).

In [ ]:
comparison = pd.DataFrame(
    {
        algorithm: {
            "roc_auc": r["roc_auc"],
            "pr_auc": r["pr_auc"],
            "brier_score": r["brier_score"],
            "passes_roc_auc": r["promotion_gate"]["passes_roc_auc"],
            "passes_pr_auc": r["promotion_gate"]["passes_pr_auc"],
            "passes_brier_score": r["promotion_gate"]["passes_brier_score"],
        }
        for algorithm, r in results.items()
    }
).T
comparison

## Calibration and operating points (best model)

In [ ]:
best_algorithm = comparison["roc_auc"].astype(float).idxmax()
best_metrics = results[best_algorithm]
print(f"Best algorithm by ROC-AUC: {best_algorithm}")

pd.DataFrame(best_metrics["calibration"])

In [ ]:
pd.DataFrame(best_metrics["operating_points"]).T

## Responsible AI preview: SHAP global importance

Same mechanism used by `src/ml_pipeline/responsible_ai/shap_explainability.py`
and `src/deployment/scoring/score.py`'s `top_factors`.

In [ ]:
from src.ml_pipeline.responsible_ai.shap_explainability import compute_shap_values, global_feature_importance

best_bundle = bundles[best_algorithm]
X_test, _ = prepare_xy(
    test_df.assign(**{LABEL_COLUMN: test_df.get(LABEL_COLUMN, 0)}),
    algorithm=best_bundle["algorithm"],
    categorical_dtypes=best_bundle["categorical_categories"],
)

shap_values = compute_shap_values(best_bundle, X_test)
importance = global_feature_importance(shap_values, list(X_test.columns))
pd.DataFrame(importance).head(10)

## Responsible AI preview: fairness metrics

Same mechanism used by `src/ml_pipeline/responsible_ai/fairness_metrics.py`,
whose `equalized_odds_difference` for `sex` and `ethnicity` feeds the
promotion gate in `src/ml_pipeline/register_model.py`.

In [ ]:
from src.ml_pipeline.responsible_ai.fairness_metrics import assess_fairness

threshold = config["evaluation"]["decision_threshold"]
fairness_report = assess_fairness(best_bundle, test_df, config, threshold)

for column, group_report in fairness_report["groups"].items():
    print(
        f"{column}: demographic_parity_difference={group_report['demographic_parity_difference']:.4f}, "
        f"equalized_odds_difference={group_report['equalized_odds_difference']:.4f}, "
        f"passes_gate={group_report['passes_equalized_odds_gate']}"
    )

## Next steps

Promote a candidate by running the full AML pipeline
(`src/ml_pipeline/pipeline_definition.py` / `pipeline_definition.yml`),
which chains validation, training, evaluation, the full responsible AI
suite, and the promotion-gated registration in
`src/ml_pipeline/register_model.py`.